# Understanding Spark's Query Execution Plans

## Table of Contents
1. [Introduction to Spark Query Execution](#1-introduction-to-spark-query-execution)
2. [Parsed Logical Plan](#2-parsed-logical-plan)
3. [Analyzed Logical Plan](#3-analyzed-logical-plan)
4. [Optimized Logical Plan](#4-optimized-logical-plan)
5. [Physical Plan](#5-physical-plan)
6. [Visualizing the Query Execution Process](#6-visualizing-the-query-execution-process)
7. [Conclusion](#7-conclusion)

## 1. Introduction to Spark Query Execution

When you submit a query to Spark, it goes through several stages of processing before the final result is computed. These stages are:

1. **Parsed Logical Plan**: Checks the syntax of the query.
2. **Analyzed Logical Plan**: Validates the existence of tables, columns, and other metadata.
3. **Optimized Logical Plan**: Applies optimization rules to improve query performance.
4. **Physical Plan**: Generates the actual execution plan that runs on the cluster.

Each of these stages plays a crucial role in ensuring that the query is executed efficiently. Let's explore each stage in detail.

## 2. Parsed Logical Plan

The **Parsed Logical Plan** is the first stage in Spark's query execution process. At this stage, Spark checks the **syntax** of the query to ensure it is valid. If there are any syntax errors, Spark will throw a `ParseException`.

### Key Points:
- **Syntax Validation**: Ensures the query follows the correct syntax.
- **Unresolved Plan**: The plan is unresolved because it does not validate the existence of tables, columns, or other metadata.

### Example:
Let's consider a simple SQL query:

```sql
SELECT * FROM orders WHERE order_status = 'CLOSED';
```

At this stage, Spark will parse the query and generate a **Parsed Logical Plan**. If the query has a syntax error (e.g., a missing semicolon), Spark will throw an error.

### Visualization:
```
Parsed Logical Plan
-------------------
'Project [*]
+- 'Filter ('order_status = 'CLOSED')
   +- 'UnresolvedRelation [orders]
```

Here, the plan is unresolved because Spark has not yet validated whether the `orders` table or the `order_status` column exists.

## 3. Analyzed Logical Plan

After the query is parsed, Spark moves to the **Analyzed Logical Plan** stage. Here, Spark validates the existence of tables, columns, and other metadata by checking the **Catalog** (a metadata repository).

### Key Points:
- **Metadata Validation**: Ensures that all tables, columns, and other entities referenced in the query exist.
- **Resolved Plan**: The plan is now resolved because all metadata has been validated.

### Example:
Continuing with the previous query:

```sql
SELECT * FROM orders WHERE order_status = 'CLOSED';
```

Spark will now check if:
- The `orders` table exists.
- The `order_status` column exists in the `orders` table.

If any of these checks fail, Spark will throw an `AnalysisException`.

### Visualization:
```
Analyzed Logical Plan
---------------------
Project [order_id#0, order_date#1, customer_id#2, order_status#3]
+- Filter (order_status#3 = 'CLOSED')
   +- SubqueryAlias orders
      +- Relation[order_id#0, order_date#1, customer_id#2, order_status#3] csv
```

Here, the plan is resolved because Spark has validated the existence of the `orders` table and the `order_status` column.

## 4. Optimized Logical Plan

Once the query is validated, Spark moves to the **Optimized Logical Plan** stage. Here, Spark applies a set of **optimization rules** to improve the query's performance.

### Key Points:
- **Optimization Rules**: Spark applies rules like **Predicate Pushdown**, **Column Pruning**, and **Constant Folding** to optimize the query.
- **Improved Performance**: The optimized plan is more efficient than the original plan.

### Example:
Continuing with the previous query:

```sql
SELECT * FROM orders WHERE order_status = 'CLOSED';
```

Spark might apply the following optimizations:
- **Predicate Pushdown**: Push the filter (`order_status = 'CLOSED'`) down to the data source to reduce the amount of data read.
- **Column Pruning**: Only read the necessary columns (`order_id`, `order_date`, `customer_id`, `order_status`) instead of all columns.

### Visualization:
```
Optimized Logical Plan
---------------------
Project [order_id#0, order_date#1, customer_id#2, order_status#3]
+- Filter (order_status#3 = 'CLOSED')
   +- Relation[order_id#0, order_date#1, customer_id#2, order_status#3] csv
```

Here, the plan is optimized to read only the necessary data and apply the filter early in the process.

## 5. Physical Plan

The final stage is the **Physical Plan**, where Spark generates the actual execution plan that will run on the cluster. This plan includes details like:
- **Join Strategies**: Whether to use Broadcast Join, Sort-Merge Join, or Shuffle Hash Join.
- **Aggregation Strategies**: Whether to use Hash Aggregate or Sort Aggregate.
- **Data Partitioning**: How data will be partitioned across the cluster.

### Key Points:
- **Execution Strategy**: The physical plan decides how the query will be executed on the cluster.
- **Performance Impact**: The choice of join and aggregation strategies can significantly impact query performance.

### Example:
Continuing with the previous query:

```sql
SELECT * FROM orders WHERE order_status = 'CLOSED';
```

Spark might generate the following physical plan:

```
Physical Plan
-------------
*(1) Project [order_id#0, order_date#1, customer_id#2, order_status#3]
+- *(1) Filter (order_status#3 = 'CLOSED')
   +- *(1) Scan csv [order_id#0, order_date#1, customer_id#2, order_status#3]
```

Here, the plan includes details like:
- **Scan**: Read the `orders` table from the CSV file.
- **Filter**: Apply the filter (`order_status = 'CLOSED'`).
- **Project**: Select the necessary columns.

## 6. Visualizing the Query Execution Process

Let's visualize the entire query execution process using a diagram:

```
Query Execution Process
-----------------------
1. Parsed Logical Plan
   - Syntax Validation
   - Unresolved Plan

2. Analyzed Logical Plan
   - Metadata Validation
   - Resolved Plan

3. Optimized Logical Plan
   - Apply Optimization Rules
   - Improved Performance

4. Physical Plan
   - Generate Execution Strategy
   - Run on Cluster
```

### Example Workflow:
1. **Parsed Logical Plan**: Check the syntax of the query.
2. **Analyzed Logical Plan**: Validate the existence of tables and columns.
3. **Optimized Logical Plan**: Apply optimizations like Predicate Pushdown and Column Pruning.
4. **Physical Plan**: Generate the execution plan and run it on the cluster.

## 7. Conclusion

In this notebook, we explored the different stages of Spark's query execution process:

1. **Parsed Logical Plan**: Validates the query syntax.
2. **Analyzed Logical Plan**: Validates the existence of tables and columns.
3. **Optimized Logical Plan**: Applies optimization rules to improve performance.
4. **Physical Plan**: Generates the actual execution plan.

### Key Takeaways:
- Each stage plays a crucial role in ensuring that the query is executed efficiently.
- Understanding these stages helps in debugging and optimizing Spark queries.
- The **Catalyst Optimizer** is responsible for transforming the logical plan into an optimized physical plan.

